In [1]:
!pip install --quiet datasets langchain_community eralchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [12]:
import pandas as pd
import sqlite3
import sqlparse
from langchain_community.utilities.sql_database import SQLDatabase
from datasets import load_dataset, Dataset

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Banco de Dados

In [4]:
def get_database_schema(db_path, qtd_ex=0):
  connection_string="sqlite:///"+db_path
  db = SQLDatabase.from_uri(connection_string, sample_rows_in_table_info=qtd_ex)
  return str(db.table_info)

In [5]:
def get_schema_dict(db_path):

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    schema_str = "{\n"

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table in tables:
        table_name=table[0]

        cursor.execute(f"PRAGMA table_info('{table_name}')")
        columns = cursor.fetchall()

        schema_str += f"  '{table_name}': ["

        for column in columns:
            column_name = column[1].replace('"','')
            schema_str += f"'{column_name}', "

        schema_str = schema_str.rstrip(", ")

        schema_str += "],\n"

    schema_str = schema_str.rstrip(",\n")
    schema_str += '\n}'

    conn.close()
    return schema_str

In [6]:
db_cnpjEN_path = '/content/drive/Shareddrives/LLMs/DSW-DataCNPJ/cnpjEN_aux.db'

In [7]:
print(get_schema_dict(db_cnpjEN_path))

{
  'age_range': ['code', 'description'],
  'city': ['code', 'name'],
  'cnae': ['code', 'name'],
  'company_size': ['code', 'description'],
  'country': ['code', 'name'],
  'legal_nature': ['code', 'description'],
  'partner_type': ['code', 'description'],
  'qualification': ['code', 'description'],
  'registration_status': ['code', 'description'],
  'registration_status_reason': ['code', 'description'],
  'establishment': ['basic_cnpj', 'order_cnpj', 'cnpj_verification_digit', 'main_or_branch', 'name', 'registration_status_code', 'data_situacao_cadastral', 'registration_status_reason_code', 'foreign_city_name', 'country_code', 'start_activity_date', 'primary_cnae_code', 'secondary_cnae_code', 'street_type', 'street_name', 'number', 'complement', 'neigborhood', 'zip_code', 'state', 'city_code', 'area_code_1', 'phone_1', 'area_code_2', 'phone_2', 'fax_area_code', 'fax', 'email', 'special_status', 'special_status_date'],
  'company': ['basic_cnpj', 'name', 'legal_nature_code', 'responsi

In [8]:
print(get_database_schema(db_cnpjEN_path))


CREATE TABLE age_range (
	code INTEGER, 
	description TEXT, 
	PRIMARY KEY (code)
)


CREATE TABLE city (
	code INTEGER, 
	name TEXT, 
	PRIMARY KEY (code)
)


CREATE TABLE cnae (
	code INTEGER, 
	name TEXT, 
	PRIMARY KEY (code)
)


CREATE TABLE company (
	basic_cnpj TEXT, 
	name TEXT, 
	legal_nature_code INTEGER, 
	responsible_qualification_code INTEGER, 
	capital FLOAT, 
	company_size_code INTEGER, 
	federative_entity_responsible TEXT, 
	PRIMARY KEY (basic_cnpj), 
	FOREIGN KEY(basic_cnpj) REFERENCES establishment (basic_cnpj), 
	FOREIGN KEY(legal_nature_code) REFERENCES legal_nature (code), 
	FOREIGN KEY(responsible_qualification_code) REFERENCES qualification (code), 
	FOREIGN KEY(company_size_code) REFERENCES company_size (code)
)


CREATE TABLE company_size (
	code INTEGER, 
	description TEXT, 
	PRIMARY KEY (code)
)


CREATE TABLE country (
	code INTEGER, 
	name TEXT, 
	PRIMARY KEY (code)
)


CREATE TABLE establishment (
	basic_cnpj TEXT, 
	order_cnpj TEXT, 
	cnpj_verification_digi

In [9]:
db_cnpjPT_path = '/content/drive/Shareddrives/LLMs/DSW-DataCNPJ/cnpjPT_aux.db'

In [10]:
print(get_schema_dict(db_cnpjPT_path))

{
  'faixa_etaria': ['codigo', 'descricao'],
  'cidade': ['codigo', 'nome'],
  'cnae': ['codigo', 'nome'],
  'porte_empresa': ['codigo', 'descricao'],
  'pais': ['codigo', 'nome'],
  'natureza_juridica': ['codigo', 'descricao'],
  'tipo_socio': ['codigo', 'descricao'],
  'qualificacao': ['codigo', 'descricao'],
  'situacao_cadastral': ['codigo', 'descricao'],
  'motivo_situacao_cadastral': ['codigo', 'descricao'],
  'estabelecimento': ['cnpj_basico', 'cnpj_ordem', 'cnpj_digito_verificador', 'matriz_ou_filial', 'nome', 'codigo_situacao_cadastral', 'data_situacao_cadastral', 'codigo_motivo_situacao_cadastral', 'nome_cidade_estrangeira', 'codigo_pais', 'data_inicio_atividade', 'codigo_cnae_principal', 'codigo_cnae_secundaria', 'tipo_logradouro', 'logradouro', 'numero', 'complemento', 'bairro', 'cep', 'estado', 'codigo_cidade', 'ddd_1', 'telefone_1', 'ddd_2', 'telefone_2', 'ddd_fax', 'fax', 'correio_eletronico', 'situacao_especial', 'data_situacao_especial'],
  'empresa': ['cnpj_basico', '

In [11]:
print(get_database_schema(db_cnpjPT_path))


CREATE TABLE cidade (
	codigo INTEGER, 
	nome TEXT, 
	PRIMARY KEY (codigo)
)


CREATE TABLE cnae (
	codigo INTEGER, 
	nome TEXT, 
	PRIMARY KEY (codigo)
)


CREATE TABLE empresa (
	cnpj_basico TEXT, 
	nome TEXT, 
	codigo_natureza_juridica INTEGER, 
	codigo_qualificacao_responsavel INTEGER, 
	capital FLOAT, 
	codigo_porte_empresa INTEGER, 
	ente_federativo_responsavel TEXT, 
	PRIMARY KEY (cnpj_basico), 
	FOREIGN KEY(cnpj_basico) REFERENCES estabelecimento (cnpj_basico), 
	FOREIGN KEY(codigo_natureza_juridica) REFERENCES natureza_juridica (codigo), 
	FOREIGN KEY(codigo_qualificacao_responsavel) REFERENCES qualificacao (codigo), 
	FOREIGN KEY(codigo_porte_empresa) REFERENCES porte_empresa (codigo)
)


CREATE TABLE estabelecimento (
	cnpj_basico TEXT, 
	cnpj_ordem TEXT, 
	cnpj_digito_verificador TEXT, 
	matriz_ou_filial INTEGER, 
	nome TEXT, 
	codigo_situacao_cadastral INTEGER, 
	data_situacao_cadastral INTEGER, 
	codigo_motivo_situacao_cadastral INTEGER, 
	nome_cidade_estrangeira TEXT, 
	

## Conjunto de Dados

In [15]:
dataCNPJ_dataset = load_dataset("NESPED-GEN/DataCNPJ", split='test')
dataCNPJ_dataset

Dataset({
    features: ['question_id', 'question_EN', 'query_cnpjEN', 'schema_linking_cnpjEN', 'question_PT', 'query_cnpjPT', 'schema_linking_cnpjPT', 'synthetic', 'hardness', '__index_level_0__'],
    num_rows: 187
})

In [16]:
dataCNPJ = dataCNPJ_dataset.to_pandas()

In [17]:
dataCNPJ.shape

(187, 10)

In [18]:
dataCNPJ['hardness'].value_counts()

,count
hardness,
extra,88
medium,44
hard,28
easy,27


In [19]:
dataCNPJ.columns

Index(['question_id', 'question_EN', 'query_cnpjEN', 'schema_linking_cnpjEN',
       'question_PT', 'query_cnpjPT', 'schema_linking_cnpjPT', 'synthetic',
       'hardness', '__index_level_0__'],
      dtype='object')

### Padronizar querys

In [26]:
import re

def replace_alias_with_table(query):
    # Expressão regular para encontrar tabelas com alias, capturando o nome da tabela e o alias
    alias_pattern = re.compile(r'(\bFROM\b|\bJOIN\b)\s+(\w+)\s+AS\s+(\w+)', re.IGNORECASE)

    # Substituições de aliases encontrados no padrão
    aliases = {match.group(3): match.group(2) for match in alias_pattern.finditer(query)}

    # Substituir cada alias pelo nome da tabela correspondente
    for alias, table in aliases.items():
        query = re.sub(r'\b' + alias + r'\b', table, query)

    # Remover 'AS' e alias das cláusulas 'FROM' e 'JOIN'
    query = re.sub(r'\bAS\s+\w+', '', query, flags=re.IGNORECASE)
    return query

In [27]:
def format_sql(query):
  return sqlparse.format(replace_alias_with_table(query), strip_whitespace=True, reindent=False, keyword_case='upper')

In [28]:
def indent_sql(format_query):
  return sqlparse.format(query, strip_whitespace=True, reindent=True, keyword_case='upper')

### Traduzir querys

In [21]:
table_map = {
    'company': 'empresa',
    'establishment': 'estabelecimento',
    'partner': 'socio',
    'taxation': 'tributacao',
    'registration_status_reason': 'motivo_situacao_cadastral',
    'city': 'cidade',
    'country': 'pais',
    'legal_nature': 'natureza_juridica',
    'qualification': 'qualificacao',
    'company_size': 'porte_empresa',
    'registration_status': 'situacao_cadastral',
    'partner_type': 'tipo_socio',
    'age_range': 'faixa_etaria'
}

column_map = {
    'code': 'codigo',
    'name': 'nome',
    'basic_cnpj': 'cnpj_basico',
    'description': 'descricao',
    'legal_nature_code': 'codigo_natureza_juridica',
    'responsible_qualification_code': 'codigo_qualificacao_responsavel',
    'company_size_code': 'codigo_porte_empresa',
    'federative_entity_responsible': 'ente_federativo_responsavel',
    'order_cnpj': 'cnpj_ordem',
    'cnpj_verification_digit': 'cnpj_digito_verificador',
    'main_or_branch': 'matriz_ou_filial',
    'registration_status_code': 'codigo_situacao_cadastral',
    'data_situacao_cadastral': 'data_situacao_cadastral',
    'registration_status_reason_code': 'codigo_motivo_situacao_cadastral',
    'foreign_city_name': 'nome_cidade_estrangeira',
    'country_code': 'codigo_pais',
    'start_activity_date': 'data_inicio_atividade',
    'primary_cnae_code': 'codigo_cnae_principal',
    'secondary_cnae_code': 'codigo_cnae_secundaria',
    'street_type': 'tipo_logradouro',
    'street_name': 'logradouro',
    'number': 'logradouro',
    'complement': 'complemento',
    'neigborhood': 'bairro',
    'zip_code': 'cep',
    'state': 'estado',
    'city_code': 'codigo_cidade',
    'area_code_1': 'ddd_1',
    'phone_1': 'telefone_1',
    'area_code_2': 'ddd_2',
    'phone_2': 'telefone_2',
    'fax_area_code': 'ddd_fax',
    'email': 'correio_eletronico',
    'special_status': 'situacao_especial',
    'special_status_date': 'data_situacao_especial',
    'partner_type_code': 'codigo_tipo_socio',
    'cpf_or_cnpj': 'cpf_ou_cnpj',
    'partner_qualification_code': 'codigo_qualificacao_socio',
    'partnership_entry_date': 'data_entrada_sociedade',
    'legal_representative_cpf': 'cpf_representante_legal',
    'legal_representative_name': 'nome_representante_legal',
    'legal_representative_qualification_code': 'codigo_qualificacao_representante_legal',
    'age_range_code': 'codigo_faixa_etaria',
    'option_for_simples_taxation': 'opcao_pelo_simples_nacional',
    'simples_taxation_option_date': 'data_opcao_simples_nacional',
    'simples_taxation_exclusion_date': 'data_exclusao_simples_nacional',
    'option_for_mei_taxation': 'opcao_pelo_mei',
    'mei_taxation_option_date': 'data_opcao_mei',
    'mei_taxation_exclusion_date': 'data_exclusao_mei'
}

In [22]:
import sqlparse
from sqlparse.sql import Identifier, IdentifierList
from sqlparse.tokens import Name

def translate_query(query):
    parsed = sqlparse.parse(query)[0]

    for token in parsed.flatten():
        if token.ttype is Name:
            value = token.value.lower()

            if value in table_map:
                token.value = table_map[value]

            elif value in column_map:
                token.value = column_map[value]

    return str(parsed)

In [23]:
query = "SELECT company.name, company.capital FROM company JOIN legal_nature ON company.legal_nature_code = legal_nature.code WHERE legal_nature.description = 'Órgão Público do Poder Executivo Federal';"
translated = translate_query(query)
print(translated)

SELECT empresa.nome, empresa.capital FROM empresa JOIN natureza_juridica ON empresa.codigo_natureza_juridica = natureza_juridica.codigo WHERE natureza_juridica.descricao = 'Órgão Público do Poder Executivo Federal';


### Arquivo test suit eval

In [29]:
with open("cnpjEN_gold.txt", "w", encoding="utf-8") as f:
    db_id = 'cnpjEN'
    for _, row in dataCNPJ.iterrows():

      query = format_sql(row['query_cnpjEN'].replace('\n',' '))

      f.write(f"{query}\t{db_id}\n")

In [31]:
with open("cnpjPT_gold.txt", "w", encoding="utf-8") as f:
    db_id = 'cnpjPT'
    for _, row in dataCNPJ.iterrows():

      query = format_sql(row['query_cnpjPT'].replace('\n',' '))

      f.write(f"{query}\t{db_id}\n")